# Trabajo Práctico I - Espacios Métricos: funciones de distancia

Gestión Avanzada de Datos - UTN FRCU

Alumno: Manuel Riedel

In [1]:
import math
import random
import unicodedata

import pandas as pd

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)

## Ejercicio 1

Calcular la Distancia de Edición entre las cadenas que siguen de acuerdo a:

- Levenshtein: inserción, eliminación y reemplazo (con el mismo costo)
- Damerau-Levenshtein: inserción, eliminación, reemplazo y transposición adyacente (con el mismo costo)

Cadenas: Horacio, Mariel, Raul, Mirta, Emanuel, Sandra, Agostina, Kesia, Yamila, Belen, Anastasia, Agustín, Luciano, Irene, Martin, Manuel, Nancy, Kenny, Marta, Casandra, Agustina, Ayelen, Yanina, Jerónimo, Juan, Julián, Cristian, León.

Las dos distancias se calculan con programación dinámica. En la celda (i, j) de la matriz queda la distancia entre las primeras i letras de una cadena y las primeras j de la otra, y el resultado es la esquina inferior derecha. Antes de comparar paso los nombres a minúscula y les saco las tildes, para que por ejemplo la "A" de Anastasia y la "a" de Casandra cuenten como la misma letra.

In [2]:
NOMBRES = ["Horacio", "Mariel", "Raul", "Mirta", "Emanuel", "Sandra", "Agostina",
           "Kesia", "Yamila", "Belen", "Anastasia", "Agustín", "Luciano", "Irene",
           "Martin", "Manuel", "Nancy", "Kenny", "Marta", "Casandra", "Agustina",
           "Ayelen", "Yanina", "Jerónimo", "Juan", "Julián", "Cristian", "León"]


def preparar(cadena):
    # minúscula y sin tildes
    sin_tildes = unicodedata.normalize("NFD", cadena)
    sin_tildes = "".join(c for c in sin_tildes if unicodedata.category(c) != "Mn")
    return sin_tildes.lower()


def levenshtein(a, b):
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i
    for j in range(len(b) + 1):
        d[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            costo = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,          # eliminación
                          d[i][j - 1] + 1,          # inserción
                          d[i - 1][j - 1] + costo)  # reemplazo
    return d[len(a)][len(b)]


def damerau_levenshtein(a, b):
    # igual que Levenshtein, más la transposición de dos letras vecinas
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i
    for j in range(len(b) + 1):
        d[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            costo = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,
                          d[i][j - 1] + 1,
                          d[i - 1][j - 1] + costo)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                d[i][j] = min(d[i][j], d[i - 2][j - 2] + 1)  # transposición
    return d[len(a)][len(b)]


def matriz_de_distancias(objetos, funcion):
    # calcula cada par una sola vez y copia el valor en la otra mitad
    n = len(objetos)
    m = [[0] * n for _ in range(n)]
    evaluaciones = 0
    for i in range(n):
        for j in range(i + 1, n):
            m[i][j] = m[j][i] = funcion(objetos[i], objetos[j])
            evaluaciones += 1
    return m, evaluaciones


nombres = [preparar(n) for n in NOMBRES]

Ejemplo con Raul y Juan, que da distinto en las dos funciones:

In [3]:
print("Levenshtein:", levenshtein("raul", "juan"))
print("Damerau-Levenshtein:", damerau_levenshtein("raul", "juan"))

Levenshtein: 4
Damerau-Levenshtein: 3


### Distancias de Levenshtein

In [4]:
lev, evaluaciones_lev = matriz_de_distancias(nombres, levenshtein)
pd.DataFrame(lev, index=NOMBRES, columns=NOMBRES)

,Horacio,Mariel,Raul,Mirta,Emanuel,Sandra,Agostina,Kesia,Yamila,Belen,Anastasia,Agustín,Luciano,Irene,Martin,Manuel,Nancy,Kenny,Marta,Casandra,Agustina,Ayelen,Yanina,Jerónimo,Juan,Julián,Cristian,León
Horacio,0,6,5,6,7,7,7,6,6,7,7,6,6,6,5,7,6,7,6,7,7,7,6,5,6,7,7,7
Mariel,6,0,4,4,3,5,7,5,4,5,8,7,6,5,3,2,5,6,3,7,7,5,4,6,6,5,7,6
Raul,5,4,0,5,4,5,8,5,4,5,8,6,6,4,5,3,4,5,4,7,7,5,5,7,4,5,7,4
Mirta,6,4,5,0,6,5,6,4,4,5,7,6,6,4,3,5,5,5,1,7,6,6,5,7,5,5,5,5
Emanuel,7,3,4,6,0,5,8,7,6,6,8,7,7,5,5,1,5,6,5,6,8,6,5,6,5,7,8,6
Sandra,7,5,5,5,5,0,7,5,4,6,7,7,7,6,5,4,4,5,4,2,7,6,3,7,5,6,7,6
Agostina,7,7,8,6,8,7,0,5,6,7,5,2,7,7,5,8,8,7,6,7,1,6,5,7,7,7,5,6
Kesia,6,5,5,4,7,5,5,0,4,4,6,5,5,5,5,6,5,3,4,6,5,5,4,6,5,4,5,4
Yamila,6,4,4,4,6,4,6,4,0,6,7,7,6,6,5,5,5,6,4,6,6,5,2,7,6,5,7,6
Belen,7,5,5,5,6,6,7,4,6,0,9,6,6,4,5,5,5,4,5,7,7,2,5,6,4,4,7,3


### Distancias de Damerau-Levenshtein

In [5]:
dam, evaluaciones_dam = matriz_de_distancias(nombres, damerau_levenshtein)
pd.DataFrame(dam, index=NOMBRES, columns=NOMBRES)

,Horacio,Mariel,Raul,Mirta,Emanuel,Sandra,Agostina,Kesia,Yamila,Belen,Anastasia,Agustín,Luciano,Irene,Martin,Manuel,Nancy,Kenny,Marta,Casandra,Agustina,Ayelen,Yanina,Jerónimo,Juan,Julián,Cristian,León
Horacio,0,6,5,6,7,7,7,6,6,7,7,6,6,6,5,7,6,7,6,7,7,7,6,5,6,7,7,7
Mariel,6,0,4,4,3,5,7,5,4,5,8,7,6,5,3,2,5,6,3,7,7,5,4,6,6,5,7,6
Raul,5,4,0,5,4,5,8,5,4,5,8,6,6,4,5,3,4,5,4,7,7,5,5,7,3,5,7,4
Mirta,6,4,5,0,6,5,6,4,4,5,7,6,6,4,3,5,5,5,1,7,6,6,5,7,5,5,5,5
Emanuel,7,3,4,6,0,5,8,7,6,6,8,7,7,5,5,1,5,6,5,6,8,6,5,6,5,7,8,6
Sandra,7,5,5,5,5,0,7,5,4,6,7,7,7,6,5,4,4,5,4,2,7,6,3,7,5,6,7,6
Agostina,7,7,8,6,8,7,0,5,6,7,5,2,7,7,5,8,8,7,6,7,1,6,5,7,7,6,4,6
Kesia,6,5,5,4,7,5,5,0,4,4,6,5,5,5,5,6,5,3,4,6,5,5,4,6,5,4,5,4
Yamila,6,4,4,4,6,4,6,4,0,6,7,7,6,6,5,5,5,6,4,6,6,5,2,7,6,5,7,6
Belen,7,5,5,5,6,6,7,4,6,0,9,6,6,4,5,5,5,4,5,7,7,2,5,6,4,4,7,3


## Ejercicio 2

1. ¿Cuáles fueron los pares de cadenas con mayor similitud (para los casos 1 y 2)? ¿Cuáles son los de mayor diferencia?
2. En su opinión, la Distancia de Edición, ¿es representativa del grado de similitud entre dos cadenas? Realice una propuesta para mejorarla.
3. Para una cantidad N de objetos, ¿qué cantidad de evaluaciones de la función es necesaria para calcular todas las distancias entre los objetos?

### 1. Pares más parecidos y más distintos

In [6]:
def pares_extremos(etiquetas, m):
    pares = [(m[i][j], etiquetas[i], etiquetas[j])
             for i in range(len(etiquetas)) for j in range(i + 1, len(etiquetas))]
    minimo = min(p[0] for p in pares)
    maximo = max(p[0] for p in pares)
    return ([p for p in pares if p[0] == minimo], [p for p in pares if p[0] == maximo])


for titulo, m in [("Levenshtein", lev), ("Damerau-Levenshtein", dam)]:
    cerca, lejos = pares_extremos(NOMBRES, m)
    print(titulo)
    print("  Más parecidos:", [f"{a} y {b} ({d})" for d, a, b in cerca])
    print("  Más distintos:", [f"{a} y {b} ({d})" for d, a, b in lejos])

Levenshtein
  Más parecidos: ['Mirta y Marta (1)', 'Emanuel y Manuel (1)', 'Agostina y Agustina (1)', 'Agustín y Agustina (1)']
  Más distintos: ['Belen y Anastasia (9)', 'Anastasia y Irene (9)', 'Anastasia y Kenny (9)', 'Anastasia y Jerónimo (9)', 'Anastasia y León (9)']
Damerau-Levenshtein
  Más parecidos: ['Mirta y Marta (1)', 'Emanuel y Manuel (1)', 'Agostina y Agustina (1)', 'Agustín y Agustina (1)']
  Más distintos: ['Belen y Anastasia (9)', 'Anastasia y Irene (9)', 'Anastasia y Kenny (9)', 'Anastasia y Jerónimo (9)', 'Anastasia y León (9)']


Pares donde Damerau-Levenshtein da menos que Levenshtein:

In [7]:
diferentes = [(NOMBRES[i], NOMBRES[j], lev[i][j], dam[i][j])
              for i in range(len(NOMBRES)) for j in range(i + 1, len(NOMBRES))
              if lev[i][j] != dam[i][j]]
pd.DataFrame(diferentes, columns=["Nombre 1", "Nombre 2", "Levenshtein", "Damerau-Levenshtein"])

,Nombre 1,Nombre 2,Levenshtein,Damerau-Levenshtein
0,Raul,Juan,4,3
1,Agostina,Julián,7,6
2,Agostina,Cristian,5,4
3,Agustina,Julián,6,5
4,Agustina,Cristian,5,4
5,Yanina,Julián,5,4
6,Yanina,Cristian,7,6


Con las dos funciones los pares más parecidos son Mirta y Marta, Emanuel y Manuel, Agostina y Agustina, y Agustín y Agustina, a distancia 1. Los más distintos son Anastasia con Belen, Irene, Kenny, Jerónimo y León, a distancia 9. Damerau-Levenshtein solo baja en 1 la distancia de 7 pares, los que tienen dos letras vecinas cruzadas, y no cambia los extremos.

### 2. ¿Es representativa?

En parte. Sirve para encontrar errores de tipeo o variantes de un mismo nombre, como Emanuel y Manuel. Pero es un valor absoluto que no tiene en cuenta el largo de las cadenas, todos los cambios cuestan lo mismo y no considera cómo suenan las letras. Por ejemplo, Juan y León están a distancia 3 cambiando 3 de 4 letras, y Anastasia y Agostina a distancia 5 cambiando 5 de 9.

Como propuesta, normalizo la distancia por el largo de la cadena más larga, así queda entre 0 y 1:

dn(a, b) = d(a, b) / max(|a|, |b|)

In [8]:
def levenshtein_normalizada(a, b):
    return levenshtein(a, b) / max(len(a), len(b))


for a, b in [("juan", "leon"), ("anastasia", "agostina")]:
    print(f"{a} y {b}: distancia {levenshtein(a, b)}, normalizada {levenshtein_normalizada(a, b):.2f}")

juan y leon: distancia 3, normalizada 0.75
anastasia y agostina: distancia 5, normalizada 0.56


Además se podrían usar costos menores para reemplazar letras que suenan parecido (b y v, c y k, s y z, i e y).

### 3. Cantidad de evaluaciones

Como d(x, x) = 0 y d(x, y) = d(y, x), alcanza con calcular cada par una sola vez: N (N − 1) / 2 evaluaciones. Para los 28 nombres son 378, que es lo que contó la función `matriz_de_distancias`.

In [9]:
N = len(NOMBRES)
print("N (N - 1) / 2 =", N * (N - 1) // 2)
print("Evaluaciones hechas:", evaluaciones_lev)

N (N - 1) / 2 = 378
Evaluaciones hechas: 378


## Ejercicio 3

Busque otras funciones de comparación de cadenas por similitud y exprese ventajas y desventajas.

| Función | Cómo funciona | Ventajas | Desventajas |
|---|---|---|---|
| Hamming | Cuenta en cuántas posiciones las letras son distintas. | Muy simple y rápida. Es métrica. | Solo sirve para cadenas del mismo largo. Una letra de más corre todas las demás. |
| LCS | Busca la subsecuencia común más larga. La distancia es \|a\| + \|b\| − 2 x LCS. | Es métrica. Sirve para comparar textos o archivos. | Un reemplazo cuesta 2. No tiene en cuenta transposiciones. |
| Jaro-Winkler | Cuenta letras que coinciden cerca de la misma posición y transposiciones, y premia el prefijo común. Da una similitud de 0 a 1. | Muy buena para nombres propios y errores de tipeo. Ya viene normalizada. | No es métrica, no sirve para índices métricos. Si el error está al principio el prefijo no ayuda. |
| q-gramas | Parte las cadenas en pedazos de q letras y compara los conjuntos (por ejemplo con Jaccard). | Sirve para textos largos y se puede acelerar con índices invertidos. | En cadenas cortas hay pocos q-gramas. Se pierde parte del orden. |
| Soundex y Metaphone | Convierten la palabra en un código según cómo suena. | Encuentran palabras que suenan igual aunque se escriban distinto. | Dependen del idioma. Solo dicen si coinciden o no. |

Implementación de algunas de ellas, probadas con pares de la lista:

In [10]:
def hamming(a, b):
    if len(a) != len(b):
        return None  # no está definida para largos distintos
    return sum(1 for x, y in zip(a, b) if x != y)


def distancia_lcs(a, b):
    l = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                l[i][j] = l[i - 1][j - 1] + 1
            else:
                l[i][j] = max(l[i - 1][j], l[i][j - 1])
    return len(a) + len(b) - 2 * l[len(a)][len(b)]


def jaro_winkler(a, b, p=0.1):
    if a == b:
        return 1.0
    ventana = max(len(a), len(b)) // 2 - 1
    usadas_a = [False] * len(a)
    usadas_b = [False] * len(b)
    m = 0
    for i, x in enumerate(a):
        for j in range(max(0, i - ventana), min(len(b), i + ventana + 1)):
            if not usadas_b[j] and b[j] == x:
                usadas_a[i] = usadas_b[j] = True
                m += 1
                break
    if m == 0:
        return 0.0
    letras_a = [x for x, u in zip(a, usadas_a) if u]
    letras_b = [y for y, u in zip(b, usadas_b) if u]
    t = sum(1 for x, y in zip(letras_a, letras_b) if x != y) / 2
    jaro = (m / len(a) + m / len(b) + (m - t) / m) / 3
    prefijo = 0
    for x, y in zip(a[:4], b[:4]):
        if x != y:
            break
        prefijo += 1
    return jaro + prefijo * p * (1 - jaro)


def jaccard_bigramas(a, b):
    ga = {a[i:i + 2] for i in range(len(a) - 1)}
    gb = {b[i:i + 2] for i in range(len(b) - 1)}
    return 1 - len(ga & gb) / len(ga | gb)


print("Prueba con el ejemplo de la teoría, martha y marhta:", round(jaro_winkler("martha", "marhta"), 3))

pares = [("mirta", "marta"), ("agustin", "agustina"), ("raul", "juan"), ("emanuel", "manuel"), ("juan", "leon")]
pd.DataFrame([(a, b, levenshtein(a, b), "-" if hamming(a, b) is None else hamming(a, b), distancia_lcs(a, b),
               round(jaro_winkler(a, b), 3), round(jaccard_bigramas(a, b), 3)) for a, b in pares],
             columns=["Cadena 1", "Cadena 2", "Levenshtein", "Hamming", "LCS",
                      "Jaro-Winkler (similitud)", "Jaccard de bigramas"])

Prueba con el ejemplo de la teoría, martha y marhta: 0.961


,Cadena 1,Cadena 2,Levenshtein,Hamming,LCS,Jaro-Winkler (similitud),Jaccard de bigramas
0,mirta,marta,1,1,2,0.880,0.667
1,agustin,agustina,1,-,1,0.975,0.143
2,raul,juan,4,4,6,0.500,1.000
3,emanuel,manuel,1,-,1,0.952,0.167
4,juan,leon,3,3,6,0.500,1.000


## Ejercicio 4

Dadas las representaciones de jugadores de handball, a través de vectores de atributos de la forma (edad, estatura, color de pelo, color de piel, peso), en la cual "color de pelo" es un valor de 1 a 10, donde 1 representa el cabello más claro y 10 el más oscuro, y la misma escala se usa para "color de piel":

(30, 1.85, 2, 4, 83), (25, 1.60, 9, 6, 65), (20, 1.80, 3, 9, 78), (28, 2.05, 4, 2, 98), (23, 1.82, 6, 8, 87), (18, 1.76, 5, 7, 84)

1. Agregar los vectores correspondientes a tres jugadores más.
2. Normalizar los vectores a 100 (en base a valores mínimos y máximos determinados por usted).
3. Calcular las distancias de Manhattan entre los vectores
4. Calcular las distancias Euclidianas entre los vectores
5. Calcular las distancias Máximas entre los vectores
6. Calcular las distancias Coseno entre los vectores
7. Calcular la media para los cuatro casos y compararlas
8. Obtener ((26, 1.75, 4, 8, 72), 25) euclidiana y ((26, 1.75, 4, 8, 72), 25) manhattan
9. Obtener el NN((23, 1.95, 3, 4, 97)) para las cuatro funciones

### 1. Jugadores agregados (J7, J8 y J9)

In [11]:
ATRIBUTOS = ["Edad", "Estatura", "Pelo", "Piel", "Peso"]

JUGADORES = [(30, 1.85, 2, 4, 83),
             (25, 1.60, 9, 6, 65),
             (20, 1.80, 3, 9, 78),
             (28, 2.05, 4, 2, 98),
             (23, 1.82, 6, 8, 87),
             (18, 1.76, 5, 7, 84),
             # los tres que agregamos
             (27, 1.90, 7, 5, 92),
             (28, 1.80, 5, 7, 75),
             (33, 1.78, 8, 10, 80)]

ETIQUETAS = [f"J{i + 1}" for i in range(len(JUGADORES))]
pd.DataFrame(JUGADORES, index=ETIQUETAS, columns=ATRIBUTOS)

,Edad,Estatura,Pelo,Piel,Peso
J1,30,1.85,2,4,83
J2,25,1.60,9,6,65
J3,20,1.80,3,9,78
J4,28,2.05,4,2,98
J5,23,1.82,6,8,87
J6,18,1.76,5,7,84
J7,27,1.90,7,5,92
J8,28,1.80,5,7,75
J9,33,1.78,8,10,80


### 2. Normalización a 100

Para la edad, la estatura y el peso elegí valores razonables para jugadores de handball, y para el pelo y la piel los extremos de la escala. Cada valor se lleva a la escala de 0 a 100 con x' = (x − mín) / (máx − mín) x 100.

In [12]:
MINIMOS = (15, 1.50, 1, 1, 55)
MAXIMOS = (40, 2.10, 10, 10, 110)


def normalizar(v, minimos, maximos):
    return tuple((x - mn) / (mx - mn) * 100 for x, mn, mx in zip(v, minimos, maximos))


base = [normalizar(j, MINIMOS, MAXIMOS) for j in JUGADORES]
pd.DataFrame(base, index=ETIQUETAS, columns=ATRIBUTOS).round(2)

,Edad,Estatura,Pelo,Piel,Peso
J1,60.0,58.33,11.11,33.33,50.91
J2,40.0,16.67,88.89,55.56,18.18
J3,20.0,50.00,22.22,88.89,41.82
J4,52.0,91.67,33.33,11.11,78.18
J5,32.0,53.33,55.56,77.78,58.18
J6,12.0,43.33,44.44,66.67,52.73
J7,48.0,66.67,66.67,44.44,67.27
J8,52.0,50.00,44.44,66.67,36.36
J9,72.0,46.67,77.78,100.00,45.45


### 3 a 6. Distancias entre los vectores

In [13]:
def manhattan(x, y):
    return sum(abs(a - b) for a, b in zip(x, y))


def euclidiana(x, y):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(x, y)))


def maxima(x, y):
    return max(abs(a - b) for a, b in zip(x, y))


def coseno(x, y):
    # distancia coseno = 1 - similitud coseno
    producto = sum(a * b for a, b in zip(x, y))
    norma_x = math.sqrt(sum(a * a for a in x))
    norma_y = math.sqrt(sum(b * b for b in y))
    return 1 - producto / (norma_x * norma_y)


FUNCIONES = [("Manhattan", manhattan), ("Euclidiana", euclidiana),
             ("Máxima", maxima), ("Coseno", coseno)]

#### Distancia Manhattan

In [14]:
m, _ = matriz_de_distancias(base, manhattan)
pd.DataFrame(m, index=ETIQUETAS, columns=ETIQUETAS).round(2)

,J1,J2,J3,J4,J5,J6,J7,J8,J9
J1,0.00,194.39,124.09,113.05,129.16,131.48,103.36,97.55,162.45
J2,194.39,0.00,176.97,247.00,140.22,144.77,140.42,119.07,144.83
J3,124.09,176.97,0.00,198.92,76.14,70.02,159.01,81.90,125.64
J4,113.05,247.00,198.92,0.00,167.22,180.45,106.58,150.15,231.06
J5,129.16,140.22,76.14,167.22,0.00,57.68,82.87,67.37,103.84
J6,131.48,144.77,70.02,180.45,57.68,0.00,118.32,63.03,137.27
J7,103.36,140.42,159.01,106.58,82.87,118.32,0.00,96.02,132.48
J8,97.55,119.07,81.90,150.15,67.37,63.03,96.02,0.00,99.09
J9,162.45,144.83,125.64,231.06,103.84,137.27,132.48,99.09,0.00


#### Distancia Euclidiana

In [15]:
m, _ = matriz_de_distancias(base, euclidiana)
pd.DataFrame(m, index=ETIQUETAS, columns=ETIQUETAS).round(2)

,J1,J2,J3,J4,J5,J6,J7,J8,J9
J1,0.00,98.74,70.44,53.91,69.37,68.95,60.75,50.67,95.91
J2,98.74,0.00,87.32,120.13,67.92,69.19,74.77,60.70,69.04
J3,70.44,87.32,0.00,101.27,40.71,34.86,75.24,45.18,77.06
J4,53.91,120.13,101.27,0.00,84.90,88.28,54.61,81.82,115.64
J5,69.37,67.92,40.71,84.90,0.00,27.87,41.84,33.68,52.86
J6,68.95,69.19,34.86,88.28,27.87,0.00,55.13,43.73,76.72
J7,60.75,74.77,75.24,54.61,41.84,55.13,0.00,47.29,68.28
J8,50.67,60.70,45.18,81.82,33.68,43.73,47.29,0.00,52.12
J9,95.91,69.04,77.06,115.64,52.86,76.72,68.28,52.12,0.00


#### Distancia Máxima

In [16]:
m, _ = matriz_de_distancias(base, maxima)
pd.DataFrame(m, index=ETIQUETAS, columns=ETIQUETAS).round(2)

,J1,J2,J3,J4,J5,J6,J7,J8,J9
J1,0.00,77.78,55.56,33.33,44.44,48.00,55.56,33.33,66.67
J2,77.78,0.00,66.67,75.00,40.00,44.44,50.00,44.44,44.44
J3,55.56,66.67,0.00,77.78,33.33,22.22,44.44,32.00,55.56
J4,33.33,75.00,77.78,0.00,66.67,55.56,33.33,55.56,88.89
J5,44.44,40.00,33.33,66.67,0.00,20.00,33.33,21.82,40.00
J6,48.00,44.44,22.22,55.56,20.00,0.00,36.00,40.00,60.00
J7,55.56,50.00,44.44,33.33,33.33,36.00,0.00,30.91,55.56
J8,33.33,44.44,32.00,55.56,21.82,40.00,30.91,0.00,33.33
J9,66.67,44.44,55.56,88.89,40.00,60.00,55.56,33.33,0.00


#### Distancia Coseno

In [17]:
m, _ = matriz_de_distancias(base, coseno)
pd.DataFrame(m, index=ETIQUETAS, columns=ETIQUETAS).round(3)

,J1,J2,J3,J4,J5,J6,J7,J8,J9
J1,0.000,0.403,0.204,0.067,0.159,0.215,0.103,0.104,0.184
J2,0.403,0.000,0.291,0.448,0.151,0.193,0.172,0.141,0.075
J3,0.204,0.291,0.000,0.315,0.050,0.047,0.175,0.079,0.106
J4,0.067,0.448,0.315,0.000,0.205,0.240,0.082,0.201,0.295
J5,0.159,0.151,0.050,0.205,0.000,0.011,0.051,0.032,0.044
J6,0.215,0.193,0.047,0.240,0.011,0.000,0.082,0.077,0.089
J7,0.103,0.172,0.175,0.082,0.051,0.082,0.000,0.062,0.093
J8,0.104,0.141,0.079,0.201,0.032,0.077,0.062,0.000,0.017
J9,0.184,0.075,0.106,0.295,0.044,0.089,0.093,0.017,0.000


### 7. Medias

In [18]:
medias = []
for nombre, funcion in FUNCIONES:
    m, _ = matriz_de_distancias(base, funcion)
    valores = [m[i][j] for i in range(len(base)) for j in range(i + 1, len(base))]
    medias.append((nombre, sum(valores) / len(valores)))

maximo_posible = {"Manhattan": 500, "Euclidiana": math.sqrt(5) * 100, "Máxima": 100, "Coseno": 1}
tabla_medias = pd.DataFrame(medias, columns=["Distancia", "Media"])
tabla_medias["Máximo posible"] = tabla_medias["Distancia"].map(maximo_posible)
tabla_medias["Media / máximo (%)"] = tabla_medias["Media"] / tabla_medias["Máximo posible"] * 100
tabla_medias.round(3)

,Distancia,Media,Máximo posible,Media / máximo (%)
0,Manhattan,129.831,500.000,25.966
1,Euclidiana,67.137,223.607,30.024
2,Máxima,47.665,100.000,47.665
3,Coseno,0.146,1.000,14.624


Para un mismo par siempre se cumple Manhattan ≥ Euclidiana ≥ Máxima, porque Manhattan suma todas las diferencias, la máxima se queda con una sola y la euclidiana está en el medio. La coseno está en otra escala (de 0 a 1) y mide el ángulo entre los vectores, no la diferencia de valores, así que para comparar las cuatro dividí cada media por el máximo posible con cinco atributos de 0 a 100.

### 8. Consulta por rango ((26, 1.75, 4, 8, 72), 25)

In [19]:
def consulta_por_rango(base, q, r, funcion):
    # todos los objetos a distancia menor o igual que r de q
    return [(ETIQUETAS[i], round(funcion(q, x), 2)) for i, x in enumerate(base) if funcion(q, x) <= r]


q1 = normalizar((26, 1.75, 4, 8, 72), MINIMOS, MAXIMOS)
print("q normalizado:", [round(x, 2) for x in q1])
print("Euclidiana:", consulta_por_rango(base, q1, 25, euclidiana))
print("Manhattan:", consulta_por_rango(base, q1, 25, manhattan))

pd.DataFrame({"Euclidiana": [euclidiana(q1, x) for x in base],
              "Manhattan": [manhattan(q1, x) for x in base]}, index=ETIQUETAS).round(2)

q normalizado: [44.0, 41.67, 33.33, 77.78, 30.91]
Euclidiana: [('J8', 20.25)]
Manhattan: []


,Euclidiana,Manhattan
J1,58.33,119.33
J2,66.21,119.51
J3,31.80,65.46
J4,96.14,171.94
J5,38.96,73.16
J6,41.83,77.71
J7,64.70,132.03
J8,20.25,44.01
J9,59.07,114.21


Con la euclidiana el resultado es J8. Con Manhattan es vacío, porque el más cercano (J8) está a 44.01. Con el mismo radio, Manhattan es más exigente porque suma todas las diferencias.

### 9. NN((23, 1.95, 3, 4, 97))

In [20]:
def vecino_mas_cercano(base, q, funcion):
    distancias = [(funcion(q, x), i) for i, x in enumerate(base)]
    d, i = min(distancias)
    return i, d


q2 = normalizar((23, 1.95, 3, 4, 97), MINIMOS, MAXIMOS)
print("q normalizado:", [round(x, 2) for x in q2])
for nombre, funcion in FUNCIONES:
    i, d = vecino_mas_cercano(base, q2, funcion)
    print(f"{nombre}: {ETIQUETAS[i]} a distancia {d:.3f}")

pd.DataFrame({nombre: [funcion(q2, x) for x in base] for nombre, funcion in FUNCIONES},
             index=ETIQUETAS).round(3)

q normalizado: [32.0, 75.0, 22.22, 33.33, 76.36]
Manhattan: J4 a distancia 71.818
Euclidiana: J4 a distancia 36.033
Máxima: J4 a distancia 22.222
Coseno: J4 a distancia 0.031


,Manhattan,Euclidiana,Máxima,Coseno
J1,81.232,42.816,28.000,0.066
J2,213.404,108.583,66.667,0.432
J3,127.101,71.055,55.556,0.186
J4,71.818,36.033,22.222,0.031
J5,117.626,62.341,44.444,0.125
J6,130.859,59.719,33.333,0.135
J7,88.980,50.068,44.444,0.073
J8,140.556,65.038,40.000,0.156
J9,221.465,104.350,66.667,0.243


## Ejercicio 5

Una empresa posee una base de datos de clientes y necesita determinar cuáles son los "mejores clientes". Para ello se requiere:

1. Definir qué características deben ser tomadas en cuenta (de cuatro a seis) y determinar su peso o importancia. Dichas características deben ser medibles cuantitativamente (por ejemplo, cantidad de días promedio de atraso en el pago de las facturas, o antigüedad del cliente).
2. Generar vectores automáticamente correspondientes a 100 clientes y normalizarlos.
3. Definir el vector consulta "mejor cliente".
4. Calcular los 2 vecinos más cercanos, utilizando la distancia euclidiana.

### 1. Características y pesos

In [21]:
# (característica, peso, mínimo, máximo, mejor valor)
CARACTERISTICAS = [("Monto comprado en el año ($)", 0.30, 50000, 5000000, "Alto"),
                   ("Cantidad de compras en el año", 0.20, 1, 60, "Alto"),
                   ("Días promedio de atraso en el pago", 0.25, 0, 90, "Bajo"),
                   ("Antigüedad como cliente (años)", 0.15, 0, 20, "Alto"),
                   ("Porcentaje de compras devueltas (%)", 0.10, 0, 15, "Bajo")]

pd.DataFrame(CARACTERISTICAS, columns=["Característica", "Peso", "Mínimo", "Máximo", "Mejor"])

,Característica,Peso,Mínimo,Máximo,Mejor
0,Monto comprado en el año ($),0.30,50000,5000000,Alto
1,Cantidad de compras en el año,0.20,1,60,Alto
2,Días promedio de atraso en el pago,0.25,0,90,Bajo
3,Antigüedad como cliente (años),0.15,0,20,Alto
4,Porcentaje de compras devueltas (%),0.10,0,15,Bajo


Al monto le di el mayor peso porque es lo que más le importa a la empresa. Después va el atraso en los pagos, porque un cliente que compra mucho pero paga tarde no es de los mejores. La cantidad de compras muestra si vuelve seguido, la antigüedad su fidelidad, y las devoluciones pesan menos pero también indican problemas.

### 2. Generación y normalización de los 100 clientes

In [22]:
def generar_clientes(cantidad, semilla=2026):
    random.seed(semilla)  # semilla fija para que siempre salgan los mismos
    clientes = []
    for _ in range(cantidad):
        monto = random.randint(50000, 5000000)
        compras = random.randint(1, 60)
        atraso = random.randint(0, 90)
        antiguedad = random.randint(0, 20)
        devoluciones = round(random.uniform(0, 15), 1)
        clientes.append((monto, compras, atraso, antiguedad, devoluciones))
    return clientes


COLUMNAS = ["Monto", "Compras", "Atraso", "Antigüedad", "Devoluciones"]
minimos = [c[2] for c in CARACTERISTICAS]
maximos = [c[3] for c in CARACTERISTICAS]
pesos = [c[1] for c in CARACTERISTICAS]

clientes = generar_clientes(100)
normalizados = [normalizar(c, minimos, maximos) for c in clientes]

tabla_clientes = pd.concat([pd.DataFrame(clientes, columns=COLUMNAS),
                            pd.DataFrame(normalizados, columns=[c + " (n)" for c in COLUMNAS]).round(2)],
                           axis=1)
tabla_clientes.index = range(1, 101)
tabla_clientes

,Monto,Compras,Atraso,Antigüedad,Devoluciones,Monto (n),Compras (n),Atraso (n),Antigüedad (n),Devoluciones (n)
1,1049250,21,64,16,9.7,20.19,33.90,71.11,80.0,64.67
2,910980,57,28,19,9.3,17.39,94.92,31.11,95.0,62.00
3,3578373,51,73,17,12.6,71.28,84.75,81.11,85.0,84.00
4,4168004,49,75,14,3.6,83.19,81.36,83.33,70.0,24.00
5,727159,8,36,3,6.7,13.68,11.86,40.00,15.0,44.67
6,4162823,44,40,6,6.0,83.09,72.88,44.44,30.0,40.00
7,2966365,60,45,12,15.0,58.92,100.00,50.00,60.0,100.00
8,4358284,41,9,10,1.3,87.04,67.80,10.00,50.0,8.67
9,4563614,19,36,14,2.1,91.18,30.51,40.00,70.0,14.00
10,4893778,20,3,11,5.5,97.85,32.20,3.33,55.0,36.67


### 3. Vector consulta "mejor cliente"

El máximo en lo que conviene que sea alto (monto, compras y antigüedad) y el mínimo en lo que conviene que sea bajo (atraso y devoluciones). Ya normalizado:

In [23]:
MEJOR_CLIENTE = (100, 100, 0, 100, 0)

### 4. Los 2 vecinos más cercanos

Uso la distancia euclidiana con pesos, para que cada característica influya según su importancia: d(x, q) = √( Σ wi (xi − qi)² ).

In [24]:
def euclidiana_con_pesos(x, y, pesos):
    return math.sqrt(sum(w * (a - b) ** 2 for a, b, w in zip(x, y, pesos)))


def vecinos_mas_cercanos(base, q, k, funcion):
    distancias = sorted((funcion(q, x), i) for i, x in enumerate(base))
    return [(i, d) for d, i in distancias[:k]]


vecinos = vecinos_mas_cercanos(normalizados, MEJOR_CLIENTE, 2,
                               lambda x, y: euclidiana_con_pesos(x, y, pesos))

resultado = tabla_clientes.loc[[i + 1 for i, _ in vecinos]].copy()
resultado["Distancia"] = [round(d, 2) for _, d in vecinos]
resultado

,Monto,Compras,Atraso,Antigüedad,Devoluciones,Monto (n),Compras (n),Atraso (n),Antigüedad (n),Devoluciones (n),Distancia
35,4177102,45,17,17,3.3,83.38,74.58,18.89,85.0,22.00,19.58
8,4358284,41,9,10,1.3,87.04,67.80,10.00,50.0,8.67,25.79


Los clientes más parecidos al mejor cliente son el 35 y el 8. El 35 compra mucho, es de los más antiguos y se atrasa poco. El 8 compra todavía más y casi no se atrasa ni devuelve, pero tiene menos antigüedad.